In [48]:
# ==========================================================
# W4D4 - FastAPI Model Serving Endpoint
# Google Colab Notebook
# ==========================================================

!pip -q install fastapi uvicorn joblib pyngrok scikit-learn nest_asyncio requests

In [49]:
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
import joblib

# Load dataset
iris = load_iris()

X = iris.data
y = iris.target

# Train model
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X, y)

# Save model
joblib.dump(model, "iris_model.pkl")

print("Model trained successfully.")

Model trained successfully.


In [50]:
%%writefile app.py

from fastapi import FastAPI
from pydantic import BaseModel
import joblib

app = FastAPI(
    title="Iris Prediction API",
    description="Week 4 Day 4 FastAPI Model Serving",
    version="1.0"
)

model = joblib.load("iris_model.pkl")

class IrisInput(BaseModel):
    sepal_length: float
    sepal_width: float
    petal_length: float
    petal_width: float


@app.get("/")
def home():
    return {
        "message": "FastAPI is running successfully!"
    }


@app.post("/predict")
def predict(data: IrisInput):

    features = [[
        data.sepal_length,
        data.sepal_width,
        data.petal_length,
        data.petal_width
    ]]

    prediction = model.predict(features)[0]

    species = {
        0: "Setosa",
        1: "Versicolor",
        2: "Virginica"
    }

    return {
        "prediction": int(prediction),
        "species": species[int(prediction)]
    }

Overwriting app.py


In [51]:
# Start FastAPI server in background

!nohup uvicorn app:app --host 0.0.0.0 --port 8000 > server.log 2>&1 &

In [53]:
from pyngrok import ngrok

# Replace with your ngrok token
ngrok.set_auth_token("3HJqKNydxgLQN6e9DdR8zQFyDuD_7Yo2ufhCrewtX8msRo12y")

# Remove old tunnels
ngrok.kill()

# Create tunnel
public_url = ngrok.connect(8000)

print("Public URL:")
print(public_url.public_url)

print("\nSwagger UI:")
print(public_url.public_url + "/docs")

Public URL:
https://trustful-shadow-bloated.ngrok-free.dev

Swagger UI:
https://trustful-shadow-bloated.ngrok-free.dev/docs


In [54]:
import time
time.sleep(5)

!cat server.log

INFO:     Started server process [12548]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


In [57]:
import requests

url = "https://trustful-shadow-bloated.ngrok-free.dev/docs"

sample = {
    "sepal_length":5.1,
    "sepal_width":3.5,
    "petal_length":1.4,
    "petal_width":0.2
}

response = requests.post(url, json=sample)

print("Status:", response.status_code)
print(response.json())

Status: 405
{'detail': 'Method Not Allowed'}
